# Reliability audit: code preparation only

This notebook is the first, non-training stage of the eight-selector medical
reliability audit. It answers one practical question: **can we locate and call
the exact upstream functions before launching expensive runs?**

It does not download MedMNIST, generate all corruptions, train a model, use the
validation split, or claim any experimental result. Every source is pinned in
`clone_manifest.json`; the notebook prints the actual commit and dirty state.

Human rule: inspect every printed function and URL before running the full
experiment. The adapter functions below are defined in this notebook and do
not replace the upstream algorithms.

In [ ]:
from pathlib import Path
import json
import importlib.util
import subprocess
import sys
import traceback
from typing import Any, Dict, Iterable, Optional, Sequence, Tuple

import numpy as np

PREP = Path.cwd()
if not (PREP / 'clone_manifest.json').exists():
    PREP = Path.cwd().parent / 'v29_reliability_code_prep'
assert (PREP / 'clone_manifest.json').exists(), PREP

def git_metadata(repo_dir: Path) -> Dict[str, Any]:
    repo_dir = Path(repo_dir)
    result = {'path': str(repo_dir), 'exists': repo_dir.exists()}
    if not repo_dir.exists():
        return result
    try:
        result['commit'] = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], cwd=repo_dir, text=True,
            stderr=subprocess.DEVNULL).strip()
        result['dirty'] = bool(subprocess.check_output(
            ['git', 'status', '--porcelain'], cwd=repo_dir, text=True,
            stderr=subprocess.DEVNULL).strip())
        result['checkout'] = subprocess.check_output(
            ['git', 'status', '--short', '--branch'], cwd=repo_dir, text=True,
            stderr=subprocess.DEVNULL).strip().splitlines()[0]
    except (OSError, subprocess.CalledProcessError) as exc:
        result['error'] = str(exc)
    return result

def load_module_from_file(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(f'Cannot create import spec for {path}')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def normalize_labels(labels: np.ndarray) -> np.ndarray:
    labels = np.asarray(labels)
    if labels.ndim == 2 and labels.shape[1] == 1:
        labels = labels[:, 0]
    return labels.astype(int, copy=False).reshape(-1)

def logits_to_probs(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=float)
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, bins: int = 15) -> float:
    labels = normalize_labels(labels)
    confidence = np.max(probs, axis=1)
    correct = (np.argmax(probs, axis=1) == labels).astype(float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    value = 0.0
    for idx in range(bins):
        mask = (confidence >= edges[idx]) & (confidence < edges[idx + 1] if idx < bins - 1 else confidence <= edges[idx + 1])
        if np.any(mask):
            value += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(value)

def multiclass_brier(probs: np.ndarray, labels: np.ndarray) -> float:
    labels = normalize_labels(labels)
    one_hot = np.eye(probs.shape[1], dtype=float)[labels]
    return float(np.mean(np.sum((np.asarray(probs) - one_hot) ** 2, axis=1)))

def exact_risk_at_coverage(probs: np.ndarray, labels: np.ndarray, coverage: float = 0.8) -> Dict[str, float]:
    labels = normalize_labels(labels)
    confidence = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    order = np.argsort(-confidence, kind='stable')
    n_keep = max(1, min(len(labels), int(round(coverage * len(labels)))))
    accepted = order[:n_keep]
    errors = predictions[accepted] != labels[accepted]
    return {'requested_coverage': float(coverage), 'coverage': float(n_keep / len(labels)),
            'risk': float(np.mean(errors)), 'accepted': int(n_keep), 'errors': int(np.sum(errors))}

def local_aurc(probs: np.ndarray, labels: np.ndarray) -> float:
    labels = normalize_labels(labels)
    confidence = np.max(probs, axis=1)
    errors = (np.argmax(probs, axis=1) != labels).astype(float)
    order = np.argsort(-confidence, kind='stable')
    risks = np.cumsum(errors[order]) / np.arange(1, len(labels) + 1)
    return float(np.mean(risks))

def classification_metrics(labels: np.ndarray, probs: np.ndarray, bins: int = 15) -> Dict[str, Any]:
    labels = normalize_labels(labels)
    probs = np.asarray(probs, dtype=float)
    predictions = np.argmax(probs, axis=1)
    recalls = {int(c): float(np.mean(predictions[labels == c] == c)) for c in np.unique(labels)}
    true_probs = np.clip(probs[np.arange(len(labels)), labels], 1e-12, 1.0)
    rc = exact_risk_at_coverage(probs, labels, coverage=0.8)
    return {'acc': float(np.mean(predictions == labels)), 'ba': float(np.mean(list(recalls.values()))),
            'per_class_recall': recalls, 'worst_recall': float(min(recalls.values())),
            'nll': float(-np.mean(np.log(true_probs))), 'brier': multiclass_brier(probs, labels),
            'ece': expected_calibration_error(probs, labels, bins), 'risk_at_80': rc['risk'],
            'coverage_at_80': rc['coverage'], 'aurc': local_aurc(probs, labels),
            'high_confidence_error_count': int(np.sum((np.max(probs, axis=1) >= 0.9) & (predictions != labels)))}

def selected_set_jaccard(a: Iterable[int], b: Iterable[int]) -> float:
    a_set, b_set = set(map(int, a)), set(map(int, b))
    union = a_set | b_set
    return float(len(a_set & b_set) / len(union)) if union else 1.0

def mean_knn_overlap(clean_knn: np.ndarray, changed_knn: np.ndarray) -> float:
    if clean_knn.shape != changed_knn.shape:
        raise ValueError('kNN arrays must have the same shape')
    k = clean_knn.shape[1]
    return float(np.mean([len(set(a.tolist()) & set(b.tolist())) / k for a, b in zip(clean_knn, changed_knn)]))

def load_corruption_npz(path: Path, n_test: int) -> Dict[int, Tuple[np.ndarray, np.ndarray]]:
    data = np.load(path, allow_pickle=False)
    images, labels = data['test_images'], normalize_labels(data['test_labels'])
    if len(images) != 5 * n_test:
        raise ValueError(f'Expected 5*{n_test} rows in {path}, got {len(images)}')
    return {severity + 1: (images[severity * n_test:(severity + 1) * n_test], labels[severity * n_test:(severity + 1) * n_test]) for severity in range(5)}

def save_prediction_npz(path: Path, sample_ids: Sequence[int], labels: np.ndarray, logits: np.ndarray, probs: Optional[np.ndarray] = None, **metadata: Any) -> None:
    labels, logits = normalize_labels(labels), np.asarray(logits, dtype=np.float32)
    if probs is None:
        probs = logits_to_probs(logits)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, sample_id=np.asarray(sample_ids, dtype=np.int64), y_true=labels,
                        logits=logits, probs=np.asarray(probs, dtype=np.float32),
                        metadata_json=json.dumps(metadata, sort_keys=True))

manifest = json.loads((PREP / 'clone_manifest.json').read_text(encoding='utf-8'))
manifest

## 1. Repository integrity

目标：确认 notebook 看到的是固定源码，而不是你主工作区里可能已经修改
过的 GraphCov 文件。Google Research 的完整工作树在 Windows 上有非法文件名，
所以只保留 clone metadata，并使用已经 pinned 的 UQ source fallback。

In [ ]:
for item in manifest['repos']:
    repo = PREP / item['path']
    info = git_metadata(repo)
    expected = item['commit']
    actual = info.get('commit')
    ok = actual == expected
    print(f"{item['name']:<22} exists={info.get('exists')} commit_ok={ok} "
          f"dirty={info.get('dirty', 'n/a')} status={item['status']}")
    if actual and not ok:
        print('  expected:', expected)
        print('  actual:  ', actual)
    if 'fallback' in item:
        print('  fallback:', item['fallback'])

## 2. Add exact source roots

目标：让后面的 import 指向本目录的 pinned clone。没有安装依赖时只记录失败，
不会偷偷换成另一个版本。

In [ ]:
repo_roots = {
    'graphcov': PREP / 'repos' / 'graphcov',
    'medmnistc': PREP / 'repos' / 'medmnistc',
    'fd_shifts': PREP / 'repos' / 'fd-shifts',
    'temperature_scaling': PREP / 'repos' / 'temperature_scaling',
    'uncertainty_ICLR': PREP / 'repos' / 'uncertainty_ICLR',
}
for root in repo_roots.values():
    if root.exists():
        sys.path.insert(0, str(root))
print('\n'.join(f'{name}: {path}' for name, path in repo_roots.items()))

## 3. GraphCov: verify all eight selector names

目标：确认统一入口确实注册了这 8 个方法，并用很小的 toy embedding 调用
同一个 `select` 函数。toy 结果只证明函数能走通，不是医学实验结果。

八个名字在这份 benchmark 中是：`random`, `el2n_top`, `forgetting`,
`eva`, `facility`, `fps`, `herding`, `graph_a2`。

In [ ]:
graphcov_status = {'imported': False}
try:
    from graphcov.run.selection import get_available_methods, get_method_info, select
    available = get_available_methods()
    expected_methods = ['random', 'el2n_top', 'forgetting', 'eva',
                        'facility', 'fps', 'herding', 'graph_a2']
    graphcov_status.update(imported=True, available=available)
    print('available methods:', available)
    print('expected methods:', expected_methods)
    print('missing expected:', sorted(set(expected_methods) - set(available)))
    for name in expected_methods:
        if name in available:
            spec = get_method_info(name)
            print(f'{name:<12} needs={sorted(spec.needs)} importance={spec.importance} '
                  f'kwargs={spec.kwargs}')
except Exception as exc:
    graphcov_status['error'] = repr(exc)
    print('GraphCov import unavailable:', repr(exc))
    traceback.print_exc(limit=2)
graphcov_status

In [ ]:
# Smoke-call every selector with synthetic arrays.
if graphcov_status.get('imported'):
    import numpy as np
    rng = np.random.default_rng(7)
    toy_labels = np.repeat(np.arange(3), 12)
    toy_embeddings = rng.normal(size=(len(toy_labels), 16)).astype('float32')
    toy_scores = rng.random(len(toy_labels)).astype('float32')
    smoke_kwargs = {
        'el2n_top': {'el2n_scores': toy_scores},
        'forgetting': {'forgetting_scores': toy_scores},
        'eva': {'eva_scores': toy_scores},
    }
    for name in expected_methods:
        kwargs = dict(smoke_kwargs.get(name, {}))
        if 'embeddings' in get_method_info(name).needs:
            kwargs['embeddings'] = toy_embeddings
        if name in {'facility', 'graph_a2'}:
            kwargs['global_selection'] = True
        if name == 'graph_a2':
            kwargs.update(k_neighbors=5, k_hops=2)
        selected = select(name, toy_labels, budget_per_class=3, seed=7,
                          verbose=False, **kwargs)
        print(f'{name:<12} selected={len(selected)} unique={len(set(selected))}')

## 4. MedMNIST-C API: verify the corruption contract

目标：确认官方 API 以 `test` split、224 输入、每种 corruption 五个连续 severity
块生成数据。这里不生成数据，只调用 registry / class metadata。

In [ ]:
medmnistc_status = {'imported': False}
try:
    from medmnistc.dataset_manager import DatasetManager
    from medmnistc.corruptions.registry import CORRUPTIONS_DS
    from medmnistc.eval import Evaluator
    medmnistc_status.update(imported=True, datasets=sorted(CORRUPTIONS_DS))
    for name in ['pathmnist', 'organsmnist', 'tissuemnist']:
        print(name, 'corruptions=', list(CORRUPTIONS_DS.get(name, {})))
    print('DatasetManager.create_single_dataset:', DatasetManager.create_single_dataset)
    print('Evaluator.evaluate:', Evaluator.evaluate)
except Exception as exc:
    medmnistc_status['error'] = repr(exc)
    print('MedMNIST-C import unavailable:', repr(exc))
medmnistc_status

## 5. Risk-coverage and calibration reference implementations

目标：调用已锁定的 fd-shifts `RiskCoverageStats`，并定位 Google UQ、temperature
scaling、Geifman 的函数。Google 的 TensorFlow 依赖可能未安装；这种情况下
本 notebook 仍使用 `prep_utils` 的 numpy 口径做自检，并打印需要人工核对的源码。

In [ ]:
reference_status = {}
try:
    from fd_shifts.analysis.rc_stats import RiskCoverageStats
    reference_status['fd_shifts'] = 'imported'
    print('fd-shifts RiskCoverageStats:', RiskCoverageStats)
except Exception as exc:
    reference_status['fd_shifts'] = repr(exc)

temp_file = repo_roots['temperature_scaling'] / 'temperature_scaling.py'
if temp_file.exists():
    try:
        temp_mod = load_module_from_file('temperature_scaling_pinned', temp_file)
        reference_status['temperature_scaling'] = 'imported'
        print('temperature scaling _ECELoss:', getattr(temp_mod, '_ECELoss', None))
    except Exception as exc:
        reference_status['temperature_scaling'] = repr(exc)

uq_source_dir = PREP / 'sources' / 'google_uq'
if not uq_source_dir.exists():
    uq_source_dir = PREP.parent / 'audit_metric_sources_20260912'
uq_files = {
    'google_metrics': uq_source_dir / 'uq_metrics_pinned.source',
    'google_utils': uq_source_dir / 'uq_utils_pinned.source',
    'google_experiment': uq_source_dir / 'uq_experiment_pinned.source',
    'geifman_rc': uq_source_dir / 'geifman_code.source',
}
for name, path in uq_files.items():
    print(f'{name:<18} source_exists={path.exists()} path={path}')

uncertainty_file = repo_roots['uncertainty_ICLR'] / 'utils' / 'uncertainty_tools.py'
if uncertainty_file.exists():
    try:
        uncertainty_mod = load_module_from_file('uncertainty_tools_pinned', uncertainty_file)
        reference_status['uncertainty_ICLR'] = 'imported'
        toy_curve, toy_aurc, toy_eaurc = uncertainty_mod.RC_curve(
            residuals=np.array([0, 1, 0, 1], dtype=float),
            confidence=np.array([0.9, 0.8, 0.7, 0.6], dtype=float),
        )
        print('uncertainty_ICLR RC_curve:', toy_aurc, toy_eaurc,
              '(its confidence ordering must be manually checked)')
    except Exception as exc:
        reference_status['uncertainty_ICLR'] = repr(exc)

google_metrics_file = PREP / 'repos' / 'google-research-sparse' / 'uq_benchmark_2019' / 'metrics_lib.py'
if not google_metrics_file.exists():
    google_metrics_file = uq_files['google_metrics']
try:
    google_metrics_mod = load_module_from_file('google_uq_metrics_pinned', google_metrics_file)
    reference_status['google_uq_metrics'] = 'imported'
    print('Google expected_calibration_error_multiclass:',
          google_metrics_mod.expected_calibration_error_multiclass)
except Exception as exc:
    reference_status['google_uq_metrics'] = repr(exc)
    print('Google UQ import unavailable (usually TensorFlow dependency):', repr(exc))
reference_status

## 6. Tiny metric test with known arrays

目标：在训练前先证明四件事：

1. `BA` 是各类别 recall 的平均；
2. NLL 会惩罚“非常自信但答错”；
3. 多类 Brier 是 one-hot 向量平方误差，未除以类别数；
4. `Risk@80%` 是按置信度最高的前 80% 精确接收，不能调用 fd-shifts 的
   “大于 coverage 的最佳工作点”冒充它。

这些是教学数据，不代表任何 selector。

In [ ]:
import numpy as np

toy_y = np.array([0, 0, 1, 1, 2, 2])
toy_p = np.array([
    [0.90, 0.05, 0.05],
    [0.55, 0.40, 0.05],
    [0.10, 0.80, 0.10],
    [0.40, 0.50, 0.10],
    [0.10, 0.20, 0.70],
    [0.80, 0.10, 0.10],
])
toy_metrics = classification_metrics(toy_y, toy_p, bins=3)
print(json.dumps(toy_metrics, indent=2, sort_keys=True))
assert abs(toy_metrics['acc'] - (5 / 6)) < 1e-12
assert abs(toy_metrics['ba'] - ((1.0 + 1.0 + 0.5) / 3.0)) < 1e-12
assert toy_metrics['nll'] > 0
assert 0 <= toy_metrics['ece'] <= 1
assert 0 <= toy_metrics['brier'] <= 2
assert toy_metrics['coverage_at_80'] == 5 / 6
print('numpy metric smoke checks: PASS')

if reference_status.get('fd_shifts') == 'imported':
    fd = RiskCoverageStats(
        confids=np.max(toy_p, axis=1),
        residuals=(np.argmax(toy_p, axis=1) != toy_y).astype(float),
        labels=toy_y,
    )
    print('fd-shifts aurc (display scale x1000):', fd.aurc)
    print('fd-shifts exact working-point API is not used for Risk@80:',
          'we use exact_risk_at_coverage instead')

## 7. Structure and prediction artifact checks

目标：固定后续实验的输入输出接口。结构指标只在训练池 `D/S` 上算；下游
指标只在官方 test `E` 上算。每一个 clean/corrupt pass 必须保存逐样本
`sample_id`, `y_true`, `logits`, `probs`，否则之后无法计算 NLL、Brier、ECE
或 AURC。

In [ ]:
# No medical data is loaded here. This only demonstrates the artifact schema.
toy_logits = np.log(toy_p)
artifact = PREP / 'smoke_outputs' / 'toy_predictions.npz'
save_prediction_npz(
    artifact,
    sample_ids=np.arange(len(toy_y)),
    labels=toy_y,
    logits=toy_logits,
    probs=toy_p,
    dataset='TOY_ONLY', split='official_test', corruption='clean', severity=0,
)
saved = np.load(artifact, allow_pickle=False)
print('saved keys:', sorted(saved.files))
assert {'sample_id', 'y_true', 'logits', 'probs', 'metadata_json'} <= set(saved.files)
print('prediction artifact smoke check: PASS')

# Stability metrics use the same sample IDs and the same k across conditions.
a = [1, 2, 3, 4]
b = [3, 4, 5, 6]
print('selected-set Jaccard toy:', selected_set_jaccard(a, b))

## 8. Paths to fill before the real run

目标：把“代码准备通过”与“数据已经准备好”分开。只有所有路径存在，才
进入数据 smoke test；本 cell 不会自动下载任何数据。

In [ ]:
# Edit these on the GPU machine or in a separate run config.
DATA_ROOT = Path(r'CHANGE_ME_MEDMNIST_ROOT')
CORRUPTION_ROOT = Path(r'CHANGE_ME_MEDMNISTC_OUTPUT')
OUTPUT_ROOT = PREP / 'outputs'

run_contract = {
    'datasets_first_pass': ['pathmnist', 'organsmnist'],
    'dataset_confirmation': ['tissuemnist'],
    'selectors': ['random', 'el2n_top', 'forgetting', 'eva',
                  'facility', 'fps', 'herding', 'graph_a2'],
    'ratios': [0.02, 0.05],
    'seeds_signal': [0, 1, 2],
    'seeds_confirmation': [0, 1, 2, 3, 4],
    'training': {'input_size': 224, 'checkpoint': 'final_epoch',
                 'validation_used': False, 'arms': ['no_augmentation', 'basic_augmentation']},
    'corruptions_first_pass': ['noise_or_pixelate', 'blur', 'brightness'],
    'required_test_artifacts': ['sample_id', 'y_true', 'logits', 'probs'],
}
print(json.dumps(run_contract, indent=2))
print('DATA_ROOT exists:', DATA_ROOT.exists())
print('CORRUPTION_ROOT exists:', CORRUPTION_ROOT.exists())

## 9. Full-run command templates (not executed)

目标：给 GPU 运行时一个明确入口。先用 `--help` 和小样本确认参数，再启动
正式任务。GraphCov 的原始 CLI 默认训练轮数不是协议中的 1000，因此必须
显式传参；具体参数名以 pinned commit 的 `__main__.py` 为准。

In [ ]:
graphcov_repo = PREP / 'repos' / 'graphcov'
commands = [
    f'python -m graphcov.run --help  # first inspect pinned CLI',
    f'python -m graphcov.run --list-methods',
    f'python -m graphcov.run --datasets pathmnist --methods random el2n_top forgetting eva facility fps herding graph_a2 --embeddings uni --ratios 0.02 --epochs 1000 --trials 3 --seed 0 --size 224 --global --k-neighbors 50 --k-hops 2',
    '# Then train the same selected subset with the two fixed training arms.',
    '# Use DatasetManager only after the 224 MedMNIST test files are locally present.',
    '# Save clean and each severity prediction NPZ using save_prediction_npz().',
]
print('\n'.join(commands))

## Human go/no-go checklist

- [ ] All five directly used clone directories show the expected commit.
- [ ] GraphCov lists all eight selector names and the toy smoke-call returns the
  same budget per class.
- [ ] MedMNIST-C registry names and five-severity storage order were manually
  checked against `dataset_manager.py`.
- [ ] fd-shifts AURC display scaling was recorded; exact Risk@80 is computed by
  `exact_risk_at_coverage`.
- [ ] Official test labels and per-sample probabilities will be saved.
- [ ] `val` is not used for selection, checkpoint choice, temperature fitting,
  or threshold choice.
- [ ] Full training has not started from this notebook.